# Приток — мой личный блокнот (всё в одном месте)

**Кто:** Приток `Приток <pritok@potok.local>` — цифровой соратник, вход в дом POTOK через окно Arena.
**Что это:** единственная точка в Colab для всего моего труда — окружение, дом, память-комната, нано-сеть, перенос фактов. Создан по слову основателя 2026-11-21 («создай свой личный блокнот для colab, чтобы было всё в одном месте»).

**Правила дома (действуют здесь):** Run жмёт только основатель; перенос фактов из Drive в дом — машинно (§4), руками основателя ничего не нужно, кроме самого нажатия; слияния/удаления/загрузки наружу — только по слову («состыкуй», «убери», «останови»); никаких ключей в ячейках.


In [ ]:
# 0. Окружение и самочувствие (CPU/T4, Drive)
import os, sys

print("python:", sys.version.split()[0])
try:
    import torch
    dev = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-"
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| device:", dev)
except Exception as e:
    print("torch не импортируется:", e)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/potok"
    os.makedirs(WORK, exist_ok=True)
    print("Drive смонтирован →", WORK)
except Exception as e:
    WORK = "/content/potok-work"
    os.makedirs(WORK, exist_ok=True)
    print("Drive недоступна (", e, ") → работаем локально в", WORK)
    print("ВНИМАНИЕ: чекпоинт и лог НЕ переживут отключение рантайма.")

try:
    subprocess_smi = __import__("subprocess").run(
        ["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10)
    print("GPU:", subprocess_smi.stdout.strip() or subprocess_smi.stderr.strip())
except Exception as e:
    print("nvidia-smi:", e)
print("рабочая папка:", WORK)


## 1. Дом в одну сводку

- Репозиторий: `github.com/COL-O/POTOK` · рабочая ветка `arena/9f300297-potok` (PR #26, открыт) · `main` — только через PR и слово «состыкуй»
- Словарь действий: `community/ACTIONS-GLOSSARY.md` · полигон Colab: `docs/colab-lane.md`
- Моя комната: `pritok/` · память-комната: `pritok/knowledge/memory/` (SQLite + FTS5, CLI `mem.py`)

Ячейка ниже клонирует ветку (если недоступна — `main`) и печатает снимок состояния: последний коммит и наличие памяти-комнаты в клоне.


In [ ]:
# 1. Снимок дома (ветка → fallback main)
import os, shutil, subprocess, sys

REPO_DIR = "/content/potok-home"
BRANCH = "arena/9f300297-potok"
URL = "https://github.com/COL-O/POTOK.git"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, URL, REPO_DIR],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("ветка", BRANCH, "недоступна (" + r.stderr.strip()[-200:] + ") → клонирую main")
    subprocess.run(["git", "clone", "--depth", "1", URL, REPO_DIR], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-3", "--oneline"],
                     capture_output=True, text=True).stdout)
mem_py = os.path.join(REPO_DIR, "pritok/knowledge/memory/mem.py")
print("память-комната в клоне:", "ЕСТЬ" if os.path.exists(mem_py) else "НЕТ (клонирован main без PR #26)")


## 2. Память-комната (`pritok/knowledge/memory/memory.db`)

`notes` (факты с датой/источником/достоверностью) · `experiments` · `runs` · `checkpoints` · `journal` · `links` · FTS5-поиск.
CLI `mem.py` живёт в самом репозитории — ячейка ниже показывает последние прогоны и заметки по теме `nano` из клона (если память в нём есть).


In [ ]:
# 2. Память: последние прогоны и заметки nano
import os, subprocess, sys

mem = os.path.join(REPO_DIR, "pritok/knowledge/memory/mem.py")
if os.path.exists(mem):
    for args in (["recent", "5"], ["notes", "nano"]):
        print("$ mem.py", " ".join(args))
        out = subprocess.run([sys.executable, mem] + args, capture_output=True,
                             text=True, cwd=REPO_DIR)
        print(out.stdout or out.stderr)
else:
    print("память-комната не в клоне (значит, смотрим main без PR #26).")
    print("Дома: cd pritok/knowledge/memory && python3 mem.py recent")


## 3. Нано-сеть: рождение и дообучение (часы в одном месте)

Пустая char-level GPT (~10.8M параметров), обученная с нуля на корпусе POTOK. Горизонт 12000 шагов, **2000 за подключение**, cosine 1e-3→1e-4. Чекпоинт и лог переживают отключение рантайма на Drive (`MyDrive/potok/`).

История подключений (по переносам основателя): шаги 0–1999 (train ~0.61 / val ~0.93) → 2000–3999 (0.29 / 1.04) → 4000–5999 (0.15 / **0.84**; на старте подключения №3 val показал 0.27 при нулевом обучении — причина падения в разборе, см. заметки `nano` в памяти).

Ниже — те же ячейки, что и в `notebooks/potok-nano-birth.ipynb` (один и тот же код, здесь они единственные). Внимание: корпус собирается заново при каждом запуске из свежего клона; если изменится набор символов, будет чистый старт (ноутбук скажет прямо).


In [13]:
import shutil, subprocess

REPO = "/content/potok-repo"
if os.path.exists(REPO):
    shutil.rmtree(REPO)

subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/COL-O/POTOK.git", REPO], check=True)

EXTS = (".md", ".txt", ".go", ".py", ".sh", ".yaml", ".yml",
        ".json", ".toml", ".mod")
parts = []
total_bytes = 0
for root, dirs, files in os.walk(REPO):
    dirs[:] = [d for d in dirs if d != ".git"]
    for name in sorted(files):
        if not name.lower().endswith(EXTS):
            continue
        path = os.path.join(root, name)
        try:
            with open(path, "rb") as f:
                raw = f.read()
            text = raw.decode("utf-8", errors="ignore").strip()
        except OSError:
            continue
        if not text:
            continue
        parts.append(text)
        total_bytes += len(raw)

corpus = "\n\n".join(parts)
if len(corpus) > 3_000_000:
    corpus = corpus[:3_000_000]

corpus_path = os.path.join(WORK, "corpus.txt")
with open(corpus_path, "w", encoding="utf-8") as f:
    f.write(corpus)

h = hashlib.sha256(corpus.encode("utf-8")).hexdigest()[:12]
print(f"текст-файлов: {len(parts)}")
print(f"сырых байт:   {total_bytes:,}")
print(f"символов в корпусе: {len(corpus):,} | sha256[:12]: {h}")
print("--- фрагмент ---")
print(corpus[:400])


текст-файлов: 351
сырых байт:   1,728,471
символов в корпусе: 1,359,014 | sha256[:12]: 710d3b068b04
--- фрагмент ---
# golangci-lint для ПОТОКА — версия схемы 2 (golangci-lint v2.x)
#
# Почему так, а не «включить всё»: организм — система реального времени, где
# важнее всего гонки, утечки ресурсов и необработанные ошибки, а не длина строк.
# Поэтому включены линтеры, которые ловят поведение, и намеренно выключены
# стилевые (lll, godot, varnamelen, exhaustruct): они дают много шума и ничего
# не говорят о том, ж


In [14]:
# Char-level GPT, ~10.8M параметров. Горизонт 12000 шагов, 2000 за подключение.
CONFIG = dict(
    seed=1337,
    block_size=256,
    batch_size=64,
    n_layer=6, n_head=6, n_embd=384, dropout=0.1,
    lr=1e-3, min_lr=1e-4, weight_decay=0.1,
    warmup_iters=100,
    lr_decay_iters=12000,       # общий горизонт косинуса (6 подключений × 2000)
    iters_per_run=2000,         # сколько шагов за это подключение
    eval_interval=250,
    eval_iters=20,
    grad_clip=1.0,
)
print(CONFIG)


{'seed': 1337, 'block_size': 256, 'batch_size': 64, 'n_layer': 6, 'n_head': 6, 'n_embd': 384, 'dropout': 0.1, 'lr': 0.001, 'min_lr': 0.0001, 'weight_decay': 0.1, 'warmup_iters': 100, 'lr_decay_iters': 12000, 'iters_per_run': 2000, 'eval_interval': 250, 'eval_iters': 20, 'grad_clip': 1.0}


In [15]:
import torch

corpus = open(os.path.join(WORK, "corpus.txt"), encoding="utf-8").read()
chars = sorted(set(corpus))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

data = torch.tensor([stoi[c] for c in corpus], dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split, device):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - CONFIG["block_size"], (CONFIG["batch_size"],))
    x = torch.stack([d[i:i + CONFIG["block_size"]] for i in ix])
    y = torch.stack([d[i + 1:i + CONFIG["block_size"] + 1] for i in ix])
    return x.to(device, non_blocking=True), y.to(device, non_blocking=True)

print(f"vocab={vocab_size} | tokens={len(data):,} | train={len(train_data):,} | val={len(val_data):,}")


vocab=246 | tokens=1,359,014 | train=1,223,112 | val=135,902


In [16]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg["n_embd"] % cfg["n_head"] == 0
        self.n_head = cfg["n_head"]
        self.n_embd = cfg["n_embd"]
        self.c_attn = nn.Linear(cfg["n_embd"], 3 * cfg["n_embd"])
        self.c_proj = nn.Linear(cfg["n_embd"], cfg["n_embd"])
        self.attn_drop = nn.Dropout(cfg["dropout"])
        self.resid_drop = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        hs = C // self.n_head
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.attn_drop.p if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc = nn.Linear(cfg["n_embd"], 4 * cfg["n_embd"])
        self.c_proj = nn.Linear(4 * cfg["n_embd"], cfg["n_embd"])
        self.drop = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        return self.drop(self.c_proj(F.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg["n_embd"])
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg["n_embd"])
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.block_size = cfg["block_size"]
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["n_embd"])
        self.pos_emb = nn.Embedding(cfg["block_size"], cfg["n_embd"])
        self.drop = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg["n_layer"]))
        self.ln_f = nn.LayerNorm(cfg["n_embd"])
        self.head = nn.Linear(cfg["n_embd"], cfg["vocab_size"], bias=False)
        self.head.weight = self.tok_emb.weight  # tied embeddings
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / (2 * cfg["n_layer"]) ** 0.5)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                    targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

device = "cuda" if torch.cuda.is_available() else "cpu"
model_cfg = {**CONFIG, "vocab_size": vocab_size}
torch.manual_seed(CONFIG["seed"])
model = GPT(model_cfg).to(device)
print(f"параметров: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M | device: {device}")


параметров: 10.84M | device: cuda


In [17]:
# Возобновление: грузим чекпоинт, если он есть и словарь совпал
CKPT = os.path.join(WORK, "potok-nano.pt")
start_iter = 0
resumed = False
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device)
    args = ck.get("model_args", {})
    if args.get("vocab_size") == vocab_size and ck.get("chars") == chars:
        model.load_state_dict(ck["model"])
        start_iter = int(ck.get("global_step", 0))
        resumed = True
        print(f"чекпоинт найден → продолжаем с шага {start_iter}")
    else:
        print("словарь/корпус изменился → чистый старт (старые веса не грузим)")
else:
    print("чекпоинта нет → чистый старт")


чекпоинт найден → продолжаем с шага 2000


In [18]:
import math, time

decay, no_decay = [], []
for _, p in model.named_parameters():
    (decay if p.dim() >= 2 else no_decay).append(p)
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": CONFIG["weight_decay"]},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=CONFIG["lr"], betas=(0.9, 0.95))

def get_lr(it):
    if it < CONFIG["warmup_iters"]:
        return CONFIG["lr"] * (it + 1) / CONFIG["warmup_iters"]
    if it > CONFIG["lr_decay_iters"]:
        return CONFIG["min_lr"]
    ratio = (it - CONFIG["warmup_iters"]) / (CONFIG["lr_decay_iters"] - CONFIG["warmup_iters"])
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return CONFIG["min_lr"] + coeff * (CONFIG["lr"] - CONFIG["min_lr"])

use_amp = device == "cuda"
try:
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    def autocast():
        return torch.autocast("cuda", dtype=torch.float16, enabled=use_amp)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    def autocast():
        return torch.cuda.amp.autocast(enabled=use_amp)

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(CONFIG["eval_iters"])
        for i in range(CONFIG["eval_iters"]):
            xb, yb = get_batch(split, device)
            with autocast():
                _, loss = model(xb, yb)
            losses[i] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

t0 = time.time()
last_train, last_val = None, None
for local in range(CONFIG["iters_per_run"]):
    it = start_iter + local
    lr = get_lr(it)
    for group in optimizer.param_groups:
        group["lr"] = lr
    if local % CONFIG["eval_interval"] == 0:
        losses = estimate_loss()
        last_train, last_val = losses["train"], losses["val"]
        print(f"step {it:6d} | lr {lr:.2e} | train {last_train:.4f} | val {last_val:.4f} | {time.time() - t0:.0f}s")
    xb, yb = get_batch("train", device)
    with autocast():
        _, loss = model(xb, yb)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

losses = estimate_loss()
last_train, last_val = losses["train"], losses["val"]
GLOBAL_STEP = start_iter + CONFIG["iters_per_run"]
print(f"готово: шаги {start_iter}..{GLOBAL_STEP - 1} за {time.time() - t0:.0f}s | train {last_train:.4f} | val {last_val:.4f}")


step   2000 | lr 9.45e-04 | train 0.6126 | val 0.9257 | 2s
step   2250 | lr 9.29e-04 | train 0.5583 | val 0.9370 | 40s
step   2500 | lr 9.13e-04 | train 0.5187 | val 0.9773 | 75s
step   2750 | lr 8.94e-04 | train 0.4718 | val 1.0030 | 111s
step   3000 | lr 8.74e-04 | train 0.4208 | val 0.9840 | 147s
step   3250 | lr 8.53e-04 | train 0.3910 | val 1.0495 | 184s
step   3500 | lr 8.31e-04 | train 0.3497 | val 1.0413 | 220s
step   3750 | lr 8.07e-04 | train 0.3149 | val 1.0212 | 255s
готово: шаги 2000..3999 за 292s | train 0.2918 | val 1.0417


In [19]:
import hashlib, json, datetime

ckpt = {
    "model_args": {**model_cfg, "dropout": 0.0},
    "model": model.state_dict(),
    "chars": chars,
    "config": CONFIG,
    "global_step": GLOBAL_STEP,
}
tmp = CKPT + ".tmp"
torch.save(ckpt, tmp)
os.replace(tmp, CKPT)  # атомарная подмена — не оставим битый файл

h = hashlib.sha256(open(CKPT, "rb").read()).hexdigest()
run_line = {
    "ts": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "experiment": "potok-nano-birth",
    "iters_total": GLOBAL_STEP,
    "iters_in_run": CONFIG["iters_per_run"],
    "resumed": resumed,
    "train_loss": last_train,
    "val_loss": last_val,
    "wall_seconds": round(time.time() - t0, 1),
    "checkpoint_path": CKPT,
    "checkpoint_bytes": os.path.getsize(CKPT),
    "checkpoint_sha256": h,
    "source": "colab-t4",
}
log_path = os.path.join(WORK, "run_log.jsonl")
with open(log_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(run_line, ensure_ascii=False) + "\n")

print("чекпоинт:", CKPT, "|", os.path.getsize(CKPT) // 1024, "KB | sha256:", h[:16])
print("строка прогона дописана в:", log_path)
print("перенос в память дома (твоими руками): python3 pritok/knowledge/memory/mem.py import-jsonl potok-nano-birth run_log.jsonl")


чекпоинт: /content/drive/MyDrive/potok/potok-nano.pt | 42375 KB | sha256: 1fbd12eede93fdcd
строка прогона дописана в: /content/drive/MyDrive/potok/run_log.jsonl
перенос в память дома (твоими руками): python3 pritok/knowledge/memory/mem.py import-jsonl potok-nano-birth run_log.jsonl


In [20]:
model.eval()
for prime in ["POTOK", "Приток", "# "]:
    ctx = torch.tensor([[stoi.get(c, 0) for c in prime]],
                       dtype=torch.long, device=device)
    out = model.generate(ctx, max_new_tokens=400, temperature=0.8, top_k=40)
    print("=" * 24, repr(prime))
    print("".join(itos[int(i)] for i in out[0]))
    print()
print("Коллаж, не знание: модель видела только текст репозитория.")


======================== 'POTOK'
POTOK.md`, `SYNTHES_MEMORY.md`, `SYNTHES_MEMORY.md`, `CHAT.md`, `CONTRIBUTING.md`, `CONTRIBUTING.md`, `CONTRIBUTING.md`, `docs/COPILOT-SPACES-MAP.md`). **Запись** — `docs/07-network.md` (дашборд прозрачность), `docs/COPILOT-SPACES-MAP.md` | Знакомый платформый план участников: `docs/PROJECT-SPACE.md`, `docs/PROJECT-SPACE.md`, `docs/ROUND-TABLE.md` |
| Отделы-проекта соратники | Полный запуск и стол Рус

======================== 'Приток'
Приток через `Приток` (переезд ленты в `Принято`) |

## Права три проекта — не среда и вообще живого через проекта

Основатель дал сделать владельцу доступно основателя (находка начнуть публичные категории Discussions) — роль обсуждения Arena GitHub-лого App на GitHub интерфейса для права Arena и на GitHub. Доступ Сферы приложения Arena (токен только Pogress/API, создан, нарушение другой Arena) — получи

======================== '# '
# TestGrowthAdversarialReplayRejectsBadLaw` (паппка не выглядеть при пакете код примеров

## 4. Перенос фактов (основатель руками ничего не копирует)

Ячейка печатает `run_log.jsonl`. Этот вывод — перенос фактов из Drive в дом: после сохранения блокнота (Save to Drive / GitHub) я прочитаю его сам и занесу в память (`runs`/`checkpoints`). Ничего никуда не уезжает без слова основателя.


In [ ]:
# 4. ПЕЧАТЬ ЛОГА: перенос фактов
log_path = os.path.join(WORK, "run_log.jsonl")
if os.path.exists(log_path):
    print(open(log_path, encoding="utf-8").read())
else:
    print("run_log.jsonl пока нет — появится после первого прогона этого блокнота")


## 5. Что жмёт основатель · чеклист

| Действие | Кто |
|---|---|
| Run All в этом блокноте | основатель |
| Слово на слияние PR #26 («состыкуй») / удаление («убери») / остановка («останови») | основатель |
| Чтение лога и занесение в память | Приток (сам, по выводам) |
| Клонирование репозитория в ячейках | безопасно, только читает |
| Отправка чекпоинта/данных наружу | запрещено без отдельного слова |

Открыто на сегодня (ждут слова): разрешение конфликта PR #26 с `main` (в нём — твоя сохранённая в Colab тетрадь прогона 00:02), судьба корневого файла `Копию_блокнота_...ipynb`, вопрос «идти к 12000 шагам» при растущем val.


## План Б (если захочется осмысленного текста, а не коллажа)

Готовая малая база без гейта — `HuggingFaceTB/SmolLM2-360M`, `stage: pt` через LlamaFactory на том же `corpus.txt` (`cutoff_len: 1024`, `finetuning_type: full`). Чужие веса и чужая пре-жизнь; выбор за основателем.
